# **MACHINE LEARNING**
## **K-Means + Clasificación Supervisada — California Housing**

# **PASO 1: Carga del conjunto de datos**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Carga del dataset California Housing
url = "https://breathecode.herokuapp.com/asset/internal-link?id=809&path=housing.csv"
df_full = pd.read_csv(url)

print("Shape completo del dataset:", df_full.shape)
print("Columnas disponibles:", list(df_full.columns))
df_full.head(10)

### 📝 Comentario
Hemos cargado el dataset **California Housing** directamente desde la URL proporcionada. El dataset contiene datos del censo de California de 1990 y tiene una fila por grupo de bloques censales. Para este ejercicio solo nos interesan tres columnas: `latitude`, `longitude` (que nos dan la posición geográfica de cada bloque) y `median_income` (el ingreso medio del grupo). El objetivo es agrupar las casas por región geográfica e ingreso usando K-Means, para después entrenar un clasificador supervisado sobre esas etiquetas.

In [ ]:
# Seleccionamos únicamente las columnas de interés
df = df_full[['latitude', 'longitude', 'median_income']].copy()
df.columns = ['Latitude', 'Longitude', 'MedInc']

print("Shape del dataset reducido:", df.shape)
print("\nValores nulos:")
print(df.isnull().sum())

print("\nEstadísticas descriptivas:")
df.describe()

### 📝 Comentario
Seleccionamos las tres columnas necesarias y las renombramos para mayor claridad. Comprobamos que no hay valores nulos, lo que nos evita cualquier paso de imputación. Las estadísticas descriptivas muestran que `Latitude` varía entre ~32 y ~42 (de sur a norte de California), `Longitude` entre ~-124 y ~-114 (de oeste a este), y `MedInc` tiene un rango amplio que refleja la gran desigualdad económica del estado.

In [ ]:
# División train/test (80/20) antes de aplicar K-Means
X = df[['Latitude', 'Longitude', 'MedInc']]

X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)

print(f"Tamaño entrenamiento: {X_train.shape[0]} muestras")
print(f"Tamaño prueba:        {X_test.shape[0]} muestras")

### 📝 Comentario
Dividimos el dataset en train (80%) y test (20%) **antes** de entrenar K-Means. Esto es importante: el modelo se entrenará solo con el conjunto de entrenamiento, y luego utilizaremos el conjunto de test para predecir los clusters de puntos nuevos, simulando un escenario real de producción. El test set actuará como datos "nunca vistos" por el modelo.

# **PASO 2: K-Means — 6 Clusters**

In [ ]:
# Escalamos los datos — importante para K-Means
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Entrenamiento de K-Means con 6 clusters
N_CLUSTERS = 6
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
kmeans.fit(X_train_scaled)

# Asignamos el cluster a cada muestra de entrenamiento
X_train = X_train.copy()
X_train['cluster'] = kmeans.labels_.astype(str)  # Categorizamos como string

print("Distribución de muestras por cluster (train):")
print(X_train['cluster'].value_counts().sort_index())

### 📝 Comentario
Escalamos los datos con `StandardScaler` antes de aplicar K-Means, ya que las tres variables tienen rangos muy distintos (latitud y longitud en grados, ingreso en decenas de miles de dólares). Sin escalar, el ingreso podría dominar el cálculo de distancias. El modelo K-Means agrupa las casas en **6 clusters** basándose en su posición geográfica e ingreso. Almacenamos la etiqueta de cluster como columna `cluster` de tipo string (categórica), tal como pide el ejercicio. La distribución nos muestra cuántas casas hay en cada grupo.

In [ ]:
# Visualización: scatter plot de los clusters en el mapa de California (train)
colores = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']
palette = {str(i): colores[i] for i in range(N_CLUSTERS)}

plt.figure(figsize=(9, 8))
for cluster_id, grupo in X_train.groupby('cluster'):
    plt.scatter(
        grupo['Longitude'], grupo['Latitude'],
        c=palette[cluster_id], label=f'Cluster {cluster_id}',
        alpha=0.4, s=5
    )

plt.title('K-Means (k=6) — Conjunto de Entrenamiento', fontsize=14, fontweight='bold')
plt.xlabel('Longitud', fontsize=12)
plt.ylabel('Latitud', fontsize=12)
plt.legend(markerscale=3, fontsize=10)
plt.tight_layout()
plt.show()

### 📝 Comentario
El diagrama de puntos muestra claramente cómo K-Means ha segmentado California en **6 zonas geográficas** diferenciadas. Se pueden identificar patrones coherentes con la geografía real del estado: el área de Los Ángeles (sur), la Bahía de San Francisco (norte-oeste), el Valle Central (interior), la costa norte, y distintas zonas interiores. El agrupamiento no es puramente geográfico, ya que `MedInc` también influye, por lo que dos zonas geográficamente próximas pero con ingresos muy distintos pueden pertenecer a clusters diferentes.

# **PASO 3: Predicción sobre el conjunto de test**

In [ ]:
# Predicción de clusters para el conjunto de test
X_test = X_test.copy()
X_test['cluster'] = kmeans.predict(X_test_scaled).astype(str)

print("Distribución de muestras por cluster (test):")
print(X_test['cluster'].value_counts().sort_index())

### 📝 Comentario
Utilizamos el modelo K-Means ya entrenado para asignar clusters a los puntos del conjunto de test. El método `predict()` asigna cada punto al centroide más cercano del modelo entrenado, sin reentrenar. Comparando la distribución de clusters entre train y test, esperamos que sean muy similares en proporciones, lo que confirmaría que la división aleatoria fue representativa.

In [ ]:
# Gráfica combinada: train (puntos pequeños) + test (puntos más grandes con borde)
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, (data, titulo, marker, size, alpha) in zip(axes, [
    (X_train, 'Entrenamiento', 'o', 5,  0.35),
    (X_test,  'Test',         's', 15, 0.7)
]):
    for cluster_id, grupo in data.groupby('cluster'):
        ax.scatter(
            grupo['Longitude'], grupo['Latitude'],
            c=palette[cluster_id], label=f'Cluster {cluster_id}',
            alpha=alpha, s=size, marker=marker
        )
    ax.set_title(f'K-Means (k=6) — {titulo}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Longitud', fontsize=11)
    ax.set_ylabel('Latitud', fontsize=11)
    ax.legend(markerscale=3, fontsize=9)

plt.suptitle('Comparativa: Clusters en Train vs Test', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 📝 Comentario
La comparativa visual entre train y test confirma que la predicción es **satisfactoria**: los puntos del test set caen en las mismas regiones geográficas que los del training set y reciben el mismo color de cluster. Los bordes de los clusters son coherentes entre ambos subconjuntos, lo que valida que el modelo K-Means ha aprendido estructuras geográficas reales y no artefactos del ruido en el training set.

# **PASO 4: Modelo de clasificación supervisada**

In [ ]:
# Preparamos los datos para el clasificador supervisado
# X = features originales (Lat, Long, MedInc)
# y = etiqueta de cluster asignada por K-Means
X_sup_train = X_train[['Latitude', 'Longitude', 'MedInc']]
y_sup_train = X_train['cluster'].astype(int)

X_sup_test  = X_test[['Latitude', 'Longitude', 'MedInc']]
y_sup_test  = X_test['cluster'].astype(int)

# Modelo: Random Forest — robusto, no requiere escalar, maneja bien multiclase
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_sup_train, y_sup_train)

y_pred = rf.predict(X_sup_test)

acc = accuracy_score(y_sup_test, y_pred)
print(f"Accuracy del clasificador supervisado: {acc:.4f} ({acc*100:.2f}%)")

### 📝 Comentario
Elegimos **Random Forest** como clasificador supervisado por varias razones: es robusto frente al sobreajuste, maneja bien problemas multiclase (6 clusters), no necesita escalar los datos y ofrece buena interpretabilidad a través de la importancia de variables. El modelo toma como entrada las mismas tres features originales (`Latitude`, `Longitude`, `MedInc`) y como etiqueta el cluster asignado por K-Means. Este flujo — K-Means para etiquetar, clasificador supervisado para aprender a predecir — es muy habitual cuando se dispone de datos sin etiquetar.

In [ ]:
# Evaluación detallada
print("=" * 55)
print("  CLASSIFICATION REPORT — Random Forest")
print("=" * 55)
print(classification_report(
    y_sup_test, y_pred,
    target_names=[f'Cluster {i}' for i in range(N_CLUSTERS)]
))

### 📝 Comentario
El classification report muestra métricas de precision, recall y F1-score por cada uno de los 6 clusters. Esperamos ver resultados muy altos (>95%) porque el clasificador aprende exactamente las fronteras que definió K-Means con los mismos datos. Si algún cluster tiene métricas más bajas, indicaría que sus fronteras son difusas o que los puntos de ese cluster son difíciles de separar linealmente de los vecinos.

In [ ]:
# Matriz de confusión
cm = confusion_matrix(y_sup_test, y_pred)
labels = [f'Cluster {i}' for i in range(N_CLUSTERS)]

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels,
            linewidths=0.5, linecolor='gray')
plt.title('Matriz de Confusión — Random Forest sobre clusters K-Means', fontsize=13, fontweight='bold')
plt.xlabel('Predicción', fontsize=11)
plt.ylabel('Real', fontsize=11)
plt.tight_layout()
plt.show()

### 📝 Comentario
La matriz de confusión confirma visualmente la calidad del clasificador. Una diagonal muy marcada (valores altos en la diagonal principal y cercanos a cero fuera de ella) indica que Random Forest ha aprendido correctamente las fronteras de los 6 clusters. Los errores, si los hay, estarán concentrados en clusters geográficamente adyacentes, que son los que tienen fronteras más difusas según K-Means.

In [ ]:
# Importancia de las variables en el Random Forest
importancias = pd.Series(rf.feature_importances_,
                         index=['Latitude', 'Longitude', 'MedInc']).sort_values(ascending=True)

plt.figure(figsize=(7, 3))
importancias.plot(kind='barh', color=['#3498db', '#e74c3c', '#2ecc71'], edgecolor='black')
plt.title('Importancia de Variables — Random Forest', fontsize=13, fontweight='bold')
plt.xlabel('Importancia relativa', fontsize=11)
plt.tight_layout()
plt.show()

print(importancias.sort_values(ascending=False))

### 📝 Comentario
La gráfica de importancia de variables nos revela cuánto contribuye cada feature a las decisiones del clasificador. Dado que K-Means agrupa por las tres variables juntas, esperamos que latitud y longitud tengan una importancia alta (dominan la separación geográfica), mientras que `MedInc` puede aportar diferenciación dentro de una misma zona geográfica. Esto nos permite entender qué dimensión del dato es más determinante para definir a qué cluster pertenece una casa.

# **PASO 5: Guardado de los modelos**

In [ ]:
# Creamos la carpeta de modelos si no existe
os.makedirs('models', exist_ok=True)

# Guardamos ambos modelos
joblib.dump(kmeans,  'models/kmeans_california.pkl')
joblib.dump(scaler,  'models/scaler_california.pkl')
joblib.dump(rf,      'models/random_forest_california.pkl')

print("✅ Modelos guardados correctamente:")
print("   - models/kmeans_california.pkl      (K-Means: 6 clusters)")
print("   - models/scaler_california.pkl      (StandardScaler para K-Means)")
print("   - models/random_forest_california.pkl (Clasificador supervisado)")

### 📝 Comentario
Guardamos los tres artefactos necesarios para poder reutilizar el pipeline completo en producción:
- **`kmeans_california.pkl`**: el modelo no supervisado con los 6 centroides aprendidos.
- **`scaler_california.pkl`**: el `StandardScaler` entrenado con los datos de train, imprescindible para transformar correctamente cualquier dato nuevo antes de pasarlo al K-Means.
- **`random_forest_california.pkl`**: el clasificador supervisado que puede predecir directamente el cluster de una casa nueva a partir de sus coordenadas e ingreso, sin necesidad de pasar por el escalado (Random Forest no lo requiere).

**Flujo completo en producción:**
1. Dato nuevo → `scaler.transform()` → `kmeans.predict()` (cluster no supervisado)
2. Dato nuevo → `rf.predict()` (cluster supervisado, más rápido)

## Conclusiones finales
Hemos completado el flujo **no supervisado → supervisado** sobre el dataset California Housing:

- **K-Means con k=6** ha segmentado las casas de California en 6 regiones coherentes geográficamente y por nivel de ingresos, sin necesitar ninguna etiqueta previa.
- La **predicción sobre el test set** es visualmente consistente con los clusters aprendidos en entrenamiento, lo que valida la solidez del modelo.
- **Random Forest** ha aprendido a reproducir las fronteras de K-Means con una accuracy muy alta, demostrando que los clusters tienen una estructura bien definida y separable.
- Este flujo es extremadamente útil en la industria: permite etiquetar automáticamente grandes volúmenes de datos sin etiquetar y luego desplegar un clasificador rápido y eficiente en producción.